# Structural Analysis

## Importing Libraries

In [1]:
import numpy as np

## Inputs

In [2]:
MAC = 0.4 # Mean Aerodynamic Chord [m]
AR = 14 # [-]
S = 5.5 # [m^2]
taper = 0.4 # [-]
rootchord = 0.5 # [m]
thicknessChordRatio = 0.17 # [-]
xAC = 0.25 # [-] position of ac with respect to the chord
# Ldstr = 
mtom = 1972 # maximum take-off mass from statistical data - Class I estimation
S1, S2 = 5.5, 5.5 # surface areas of wing one and two
A = 14 # aspect ratio of a wing, not aircraft
nmax = 3.2 # maximum load factor
n_ult = nmax*1.5 # 3.2 is the max we found, 1.5 is the safety factor
Pmax = 15.25 # this is defined as maximum perimeter in Roskam, so i took top down view of the fuselage perimeter
lf = 7.2 # length of fuselage
m_pax = 95 # average mass of a passenger according to Google
n_prop = 16 # number of engines
n_pax = 5 # number of passengers (pilot included)
pos_fus = 3.6 # fuselage centre of mass away from the nose
pos_lgear = 3.6 # landing gear position away from the nose
pos_frontwing, pos_backwing = 0.2, 7 # positions of the wings away from the nose
m_prop = [30]*16 # list of mass of engines (so 30 kg per engine with nacelle and propeller)
pos_prop = [0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0] # 8 on front wing and 8 on back wing

## Optimized

In [3]:
thicknessOfSkin = 1e-3 # [m]
thicknessOfSpar = 1e-2 # [m]
nStrT = 0  # [-] number of stringers on the top
nStrB = 0 # [-] number of stringers on the bottom
StrA = 0 # [m^2] stringer cross-sectional area

## Wing Structure Geometry

In [13]:
from Geometry import HatStringer, JStringer, ZStringer, WingBox, WingStructure
from SolveLoads import WingLoads, Engines, Fatigue
from Weight import *

base, height = 0.75 - 0.15, 0.11571117509557907 + 0.03145738125495376 # x/c, y/c


print('normalBox =', normalBox := WingBox(thicknessOfSkin, thicknessOfSpar, base, height))

print('wing =', wing := WingStructure(span := (AR * S) ** 0.5, taper, rootchord, normalBox))

args = dict(span=span, taper=taper, cr=rootchord, tsk=thicknessOfSkin, tsp=thicknessOfSpar,
            toc=thicknessChordRatio, nStrT=nStrT, nStrB=nStrB, StrA=StrA, mac=MAC, xac=xAC,
            engines=Engines(2960, 2960, list(np.linspace(0.1*span/2, 0.8*span/2, 4)), 400 * 9.81 / 8),
           frac=0.6)

loads = WingLoads(**args)

wing = Wing(mtom, S1, S2, n_ult, A, [pos_frontwing, pos_backwing])
fuselage = Fuselage(mtom, Pmax, lf, n_pax, pos_fus)
lgear = LandingGear(mtom, pos_lgear)
props = Propulsion(n_prop, m_prop, pos_prop)
w = Weight(m_pax, wing, fuselage, lgear, props, cargo_m = 85, cargo_pos = 6, battery_m = 400, battery_pos = 3.6, p_pax = [1.5, 3, 3, 4.2, 4.2])

wingWeight = w.wing.mass[0] * 9.81
lift = nmax * 1.5 * w.mtom * 9.81
drag = lift / 19.03
pos = np.linspace(0, span/2, 10000)
dragd = 2 * drag / (np.pi * span) * np.sqrt(1 - 4 * np.power(pos / span, 2))
liftd = 2 * lift / (np.pi * span) * np.sqrt(1 - 4 * np.power(pos / span, 2))
Mac = 0.0645 * 0.5 * 1.2 * 53 ** 2 * 5.25 * 0.65

reactionsCruise = loads.equilibriumCruise([pos, dragd], [pos, liftd], [pos, [Mac / span]*len(pos)], wingWeight)
reactionsVTO = loads.equilibriumVTO(wingWeight)
lift, wgt = loads.internalLoads([pos, dragd], [pos, liftd], [pos, [Mac / span]*len(pos)], wingWeight)
VxVTO, MyVTO = loads.internalLoadsVTO(wingWeight)

normalBox = Wingbox(Height=0.14716855635053283, Base=0.6, Tsk = 0.001, Tsp = 0.01, Stringers = 0)
wing = WingStructure(span=8.774964387392123, taper=0.4, rc=0.5, tc=0.2, box=Wingbox(Height=0.14716855635053283, Base=0.6, Tsk = 0.001, Tsp = 0.01, Stringers = 0))


In [14]:
from Draw import InternalLoading
InternalLoading(0, span/2, **{l: loads[l] for l in 'T, My, Mx, Vx, Vy'.split(', ')}).show(renderer='iframe')

In [16]:
InternalLoading(0, span/2, Vx = VxVTO, My = MyVTO).show(renderer='iframe')

In [20]:
loads.stresses()

(448076187.1506728,
 4632740.544279172,
 1292482516.9750552,
 16421914.513021462,
 1509983592.1799433,
 16642051.955525106)